## Step 1 — Input Sentence


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments
from transformers import pipeline
import torch
import pandas as pd
from sklearn.model_selection import train_test_split

reviews_df = pd.read_csv("data/output/processed/reviews.csv")

print(reviews_df.head())
print(reviews_df.columns)

sample_review = reviews_df['review_text'].iloc[0]
print(f"\nSample Input Sentence:\n'{sample_review}'")

  review_id meal_id            meal_name      restaurant  rating  \
0   R000001   M0001  Classic Beef Burger  Mama's Kitchen     1.9   
1   R000002   M0001  Classic Beef Burger  Mama's Kitchen     3.8   
2   R000003   M0001  Classic Beef Burger  Mama's Kitchen     4.0   
3   R000004   M0001  Classic Beef Burger  Mama's Kitchen     4.7   
4   R000005   M0001  Classic Beef Burger  Mama's Kitchen     4.0   

                                         review_text sentiment_label  \
0  Disappointed with the quality. Not what I expe...        negative   
1  Pretty good, I've had better but wouldn't say ...         neutral   
2  Super fast delivery and the food was outstandi...        positive   
3  Perfect seasoning and amazing presentation. Wo...        positive   
4  Loved every bite. The packaging was also excel...        positive   

  review_date  helpful_votes  
0  2026-03-19              8  
1  2026-04-05             44  
2  2026-02-01             50  
3  2025-12-14             10  
4  

## Step 2 — Tokenization


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

sample = "The food was not bad"
tokens = tokenizer.tokenize(f"[CLS] {sample} [SEP]")
print("Tokens:", tokens)

encoded = tokenizer(sample, return_tensors='pt')
print("Token IDs:", encoded['input_ids'])
print("Decoded:", tokenizer.decode(encoded['input_ids'][0]))

Tokens: ['[CLS]', 'the', 'food', 'was', 'not', 'bad', '[SEP]']
Token IDs: tensor([[ 101, 1996, 2833, 2001, 2025, 2919,  102]])
Decoded: [CLS] the food was not bad [SEP]


## Step 3 — Token Embedding
Each token is converted into a **dense vector** (list of numbers) that BERT can process.
Semantically similar words will have numerically close vectors.

Example: `bad → [0.80, 0.90, ...]`

BERT learns these embeddings during pre-training on large corpora.

## Step 4 — Position Embedding
BERT adds **positional information** to each token so it knows word order.

This is critical because:
- *"dog bites man"* ≠ *"man bites dog"* — same words, different meaning.

Each position index gets a unique vector added to the token embedding.

## Step 5 — Segment Embedding
BERT supports two-sentence tasks (e.g., Question Answering). A **segment embedding** tells the model which sentence each token belongs to (Sentence A or Sentence B).

For single-sentence classification (our task), all tokens belong to **Segment A**.

## Step 6 — Create Initial Input Vector



In [ ]:
sentiment_pipe = pipeline("sentiment-analysis")

def get_label(text):
    if pd.isna(text):
        return 1  # neutral
    result = sentiment_pipe(str(text)[:512])[0]['label']
    if result == "NEGATIVE":
        return 0
    else:
        return 2  

reviews_df['label'] = reviews_df['review_text'].apply(get_label)
print("Label distribution:")
print(reviews_df['label'].value_counts())

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Label distribution:
label
2    1273
0     562
Name: count, dtype: int64


## Step 7 — Why Attention?
Words have **different meanings depending on context**. Attention lets BERT focus on the most relevant surrounding words.

Example:
- `"bad"` → Negative
- `"not bad"` → Usually Positive

Without attention, BERT would miss that `"not"` reverses the meaning of `"bad"`.

## Step 8 — Build Query, Key, Value


In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    reviews_df['review_text'].tolist(),
    reviews_df['label'].tolist(),
    test_size=0.2,
    random_state=42
)

print(f"Train size: {len(train_texts)} | Test size: {len(test_texts)}")

Train size: 1468 | Test size: 367


## Step 9 — Calculate Attention Scores


In [ ]:
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

print("Tokenization complete. Keys:", list(train_encodings.keys()))

Tokenization complete. Keys: ['input_ids', 'token_type_ids', 'attention_mask']


## Step 10 — Raw Attention Scores
The dot-product scores are **raw and unnormalized**.

Example scores for the word `"bad"` attending to others:
```
not  → 0.97
bad  → 1.67
food → 0.32
The  → 0.11
```

Higher score = stronger influence. These get passed through Softmax next.

## Step 11 — Softmax Attention Weights
Raw scores are converted to **probabilities** using Softmax so they sum to 1:

$$\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Example after Softmax:
```
not  → 0.35
bad  → 0.50
food → 0.10
The  → 0.05
```
Now we know the **relative importance** of each word to the current token.

In [ ]:
class ReviewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = ReviewsDataset(train_encodings, train_labels)
test_dataset  = ReviewsDataset(test_encodings,  test_labels)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Test  dataset: {len(test_dataset)}  samples")

Train dataset: 1468 samples
Test  dataset: 367  samples


## Step 12 — Build Contextual Vector
Each token's new representation is a **weighted sum of all Value vectors**, using the Softmax attention weights:

$$\text{Context Vector} = \sum (\text{attention weight} \times \text{Value Vector})$$

This is what makes BERT powerful — `"bad"` now carries information from `"not"`, making:
```
bad ≠ not bad
```

## Step 13 — Contextual Vectors for All Tokens
Steps 8–12 are applied to **every token** in parallel across all 12 Transformer layers.

After all layers:
- `"bad"` has absorbed context from `"not"`
- `"food"` has absorbed context from `"delivery"`, `"packaging"`, etc.
- Every token is now **context-aware**.

## Step 14 — [CLS] Vector



In [ ]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3  
)

print("Model loaded: bert-base-uncased with 3-class classification head")
print(f"[CLS] vector size: {model.config.hidden_size} dimensions")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: bert-base-uncased with 3-class classification head
[CLS] vector size: 768 dimensions


## Step 15 — Classification Layer
The `[CLS]` vector is passed through a **linear (dense) layer** to produce class scores:

$$z = W \cdot h_{[CLS]} + b$$

- **W** — learned weight matrix (768 × 3)
- **b** — bias vector
- **z** — raw logits for each class (Negative, Neutral, Positive)

This layer is randomly initialized and **trained on our food review data**.

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
c:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=368, training_loss=0.036749565083047615, metrics={'train_runtime': 668.9096, 'train_samples_per_second': 4.389, 'train_steps_per_second': 0.55, 'total_flos': 34702193110032.0, 'train_loss': 0.036749565083047615, 'epoch': 2.0})

## Step 16 — Compute Class Scores (Logits)


In [ ]:
eval_results = trainer.evaluate()
print("Evaluation Results:", eval_results)

c:\Users\HP\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
No log,0.000287,368


Evaluation Results: {'eval_loss': 0.00028673248016275465}


## Step 17 — Convert Scores to Probabilities & Final Prediction
Logits are passed through **Softmax** to get class probabilities (sum to 1):

Example:
```
Negative = 0.40
Neutral  = 0.20
Positive = 0.60   ← argmax → Final Prediction: POSITIVE
```

We use `torch.argmax` to select the class with the highest probability.

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    outputs = model(**inputs)               
    probs   = torch.softmax(outputs.logits, dim=1)  
    pred    = torch.argmax(probs, dim=1).item()      

    if pred == 0:
        return "NEGATIVE", -1
    elif pred == 1:
        return "NEUTRAL", 0
    else:
        return "POSITIVE", 1

results = reviews_df['review_text'].apply(predict_sentiment)

reviews_df['sentiment']       = results.apply(lambda x: x[0])
reviews_df['sentiment_score'] = results.apply(lambda x: x[1])

print("Sentiment distribution:")
print(reviews_df['sentiment'].value_counts())

Sentiment distribution:
sentiment
POSITIVE    1273
NEGATIVE     562
Name: count, dtype: int64


In [ ]:
reviews_df = reviews_df[
    ['review_id', 'meal_id', 'review_text', 'sentiment', 'sentiment_score']
]

reviews_df.to_csv("data/output/processed/reviews_with_sentiment.csv", index=False)
print("Saved: data/output/processed/reviews_with_sentiment.csv")
print(reviews_df.head())

Saved: data/output/processed/reviews_with_sentiment.csv
  review_id meal_id                                        review_text  \
0   R000001   M0001  Disappointed with the quality. Not what I expe...   
1   R000002   M0001  Pretty good, I've had better but wouldn't say ...   
2   R000003   M0001  Super fast delivery and the food was outstandi...   
3   R000004   M0001  Perfect seasoning and amazing presentation. Wo...   
4   R000005   M0001  Loved every bite. The packaging was also excel...   

  sentiment  sentiment_score  
0  NEGATIVE               -1  
1  NEGATIVE               -1  
2  POSITIVE                1  
3  POSITIVE                1  
4  POSITIVE                1  
